# Seq2Seq 모델 Q&A Chatbot 구현

1. QnA 데이터셋을 찾아서 처리해서 준비한다. (전처리 전반)
- input을 이용해서 input을 받고 encoder -> decoder -> output 
- 좋은 아침이야, 오늘 점심 뭐 먹을까? -> input / output이 반복되는 구조만 나오면 될 듯.
2. Encoder, Decoder를 만들고, 그걸로 Seq2Seq(Encoder + Decoder) 모델을 만든다.
3. 1에서 준비한 데이터로 2에서 만든 모델을 학습시킨다.
4. Chatbot을 만든다. (모델 추론 + while문)

In [11]:
import re
from collections import Counter
import zipfile
import glob
import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [12]:
zip_path = r"./3.개방데이터/1.데이터/Training/01.원천데이터/TS_004. 일반군_0001. 1회기.zip"

with zipfile.ZipFile(zip_path, 'r') as z:
    file_list = z.namelist()

with zipfile.ZipFile(zip_path, 'r') as z:
    sample_file = file_list[0]
    text = z.read(sample_file).decode("utf-8")

print(text[:400])

상담사 : 저는 상담사 @COUNSELOR이라고 하고요.
내담자 : 네.
상담사 : 여기서 10년째 대학생들 만나서 이렇게 상담을 진행하고 있고 이거는 이렇게 참여하게 되셔서, 8회기 진행할 겁니다. 8회기를 진행할 거고 1시간 혹은 좀 더 길어지면 1시간 반 이 정도 안쪽으로 상담을 쭉 진행을 하는 식의 프로그램이고 사실 어디 특별한 정서적인 영역으로 뜨지는 사실 않으셔서 그거를 그게 중요한 게 아니라 그거 외에 혹시 상담을 받거나 다루고 싶은 게 있으면 그런 것들을 다루면서 진행을 해 볼 겁니다. 혹시 사업에 대해서 궁금하신.
내담자 : 없어요.
상담사 : 무슨 과?
내담자 : @DEPARTMENT.
상담사 : @DEPARTMENT.
내담자 : 몇 @YEAR?
상담사 : @YEAR.
내담자 : @YEAR


In [13]:
# zip 파일 경로
zip_paths = sorted(glob.glob("./3.개방데이터/1.데이터/Training/01.원천데이터/*.zip"))

pairs = []

def parse_lines(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    
    dialogues = []
    current_speaker = None
    current_utterance = []

    for line in lines:
        m = re.match(r"^(상담사|내담자)\s*:\s*(.*)$", line)
        
        if m:
            if current_speaker is not None:
                dialogues.append((current_speaker, " ".join(current_utterance).strip()))
            
            current_speaker = m.group(1)
            current_utterance = [m.group(2).strip()]
        else:
            # 줄바꿈으로 이어진 발화는 이전 화자 발화에 붙이기
            if current_speaker is not None:
                current_utterance.append(line)

    # 마지막 발화 저장
    if current_speaker is not None:
        dialogues.append((current_speaker, " ".join(current_utterance).strip()))
    
    return dialogues

def merge_same_speaker(dialogues):
    if not dialogues:
        return []
    
    merged = [list(dialogues[0])]
    
    for speaker, utterance in dialogues[1:]:
        if speaker == merged[-1][0]:
            merged[-1][1] += " " + utterance
        else:
            merged.append([speaker, utterance])
    
    return [(speaker, utterance.strip()) for speaker, utterance in merged]

for zip_path in zip_paths:
    zip_name = os.path.basename(zip_path)
    
    with zipfile.ZipFile(zip_path, "r") as z:
        for file_name in z.namelist():
            if not file_name.lower().endswith(".txt"):
                continue
            
            try:
                raw = z.read(file_name)
                try:
                    text = raw.decode("utf-8")
                except UnicodeDecodeError:
                    text = raw.decode("cp949")
            except Exception as e:
                print(f"읽기 실패: {zip_name} / {file_name} / {e}")
                continue

            dialogues = parse_lines(text)
            dialogues = merge_same_speaker(dialogues)

            for i in range(len(dialogues) - 1):
                speaker1, utt1 = dialogues[i]
                speaker2, utt2 = dialogues[i + 1]

                if speaker1 == "상담사" and speaker2 == "내담자":
                    pairs.append({
                        "source_zip": zip_name,
                        "source_txt": file_name,
                        "Q": utt1,
                        "A": utt2
                    })

# 데이터프레임 생성
df = pd.DataFrame(pairs)
df.head()

,source_zip,source_txt,Q,A
0,TS_004. 일반군_0001. 1회기.zip,/resource_normal_1_check_N008.txt,저는 상담사 @COUNSELOR이라고 하고요.,네.
1,TS_004. 일반군_0001. 1회기.zip,/resource_normal_1_check_N008.txt,여기서 10년째 대학생들 만나서 이렇게 상담을 진행하고 있고 이거는 이렇게 참여하게...,없어요.
2,TS_004. 일반군_0001. 1회기.zip,/resource_normal_1_check_N008.txt,무슨 과?,@DEPARTMENT.
3,TS_004. 일반군_0001. 1회기.zip,/resource_normal_1_check_N008.txt,@DEPARTMENT.,몇 @YEAR?
4,TS_004. 일반군_0001. 1회기.zip,/resource_normal_1_check_N008.txt,@YEAR.,@YEAR이시고.


In [14]:
df = df[['Q', 'A']]
df.head()

,Q,A
0,저는 상담사 @COUNSELOR이라고 하고요.,네.
1,여기서 10년째 대학생들 만나서 이렇게 상담을 진행하고 있고 이거는 이렇게 참여하게...,없어요.
2,무슨 과?,@DEPARTMENT.
3,@DEPARTMENT.,몇 @YEAR?
4,@YEAR.,@YEAR이시고.


In [15]:
# @XXXX와 같은 개인정보가 마스킹 되어있는 것들을 한국어로 대체
text_all = ' '.join(
    df
    .fillna('')
    .astype(str)
    .values
    .ravel()
)

tokens = re.findall(r'@[A-Z_]+', text_all)
token_counts = Counter(tokens)

token_df = pd.DataFrame(
    token_counts.items(),
    columns=['token', 'count']
).sort_values('count', ascending=False)

print(token_df)

           token  count
9          @NAME   3585
8           @ETC   1781
4         @TITLE   1613
7         @PLACE   1131
13         @TIME    563
2          @YEAR    493
6        @SCHOOL    443
3           @AGE    442
11         @DATE    336
1    @DEPARTMENT    303
15  @FAMILY_NAME    260
12      @COMPANY     90
5         @EVENT     87
20       @COURSE     67
10    @PROFESSOR     64
0     @COUNSELOR     32
21   @STUDENT_ID     18
16         @CLUB     11
14        @PHONE      6
19      @ADDRESS      5
17     @HOSPITAL      4
18      @PROJECT      3
22          @DOB      3


In [16]:
# 하이퍼파라미터 설정
BATCH_SIZE = 64
MAX_VOCAB_SIZE = 10000
EMBEDDING_DIM = 100
LATENT_DIM = 128
EPOCHS = 100

In [17]:
import pandas as pd

df['Q'] = df['Q'].astype(str).str.strip()
df['A'] = df['A'].astype(str).str.strip()

encoder_texts = df['Q'].tolist()
decoder_input_texts = ['<sos> ' + a for a in df['A']]
decoder_target_texts = [a + ' <eos>' for a in df['A']]

print(len(encoder_texts), len(decoder_input_texts), len(decoder_target_texts))
print(encoder_texts[0])
print(decoder_input_texts[0])
print(decoder_target_texts[0])

24058 24058 24058
저는 상담사 @COUNSELOR이라고 하고요.
<sos> 네.
네. <eos>


In [18]:
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, filters='')
tokenizer.fit_on_texts(encoder_texts + decoder_input_texts + decoder_target_texts)

encoder_seqs = tokenizer.texts_to_sequences(encoder_texts)
decoder_input_seqs = tokenizer.texts_to_sequences(decoder_input_texts)
decoder_target_seqs = tokenizer.texts_to_sequences(decoder_target_texts)

num_words = min(MAX_VOCAB_SIZE, len(tokenizer.word_index) + 1)

max_encoder_len = max(len(s) for s in encoder_seqs)
max_decoder_len = max(len(s) for s in decoder_input_seqs)

print(f'num_words = {num_words}')
print(f'max_encoder_len = {max_encoder_len}')
print(f'max_decoder_len = {max_decoder_len}')

num_words = 10000
max_encoder_len = 1577
max_decoder_len = 2331


In [9]:
# 패딩 처리
encoder_input_data = pad_sequences(
    encoder_seqs,
    maxlen=max_encoder_len,
    padding='post'
)

decoder_input_data = pad_sequences(
    decoder_input_seqs,
    maxlen=max_decoder_len,
    padding='post'
)

decoder_target_data = pad_sequences(
    decoder_target_seqs,
    maxlen=max_decoder_len,
    padding='post'
)

print(encoder_input_data.shape)
print(decoder_input_data.shape)
print(decoder_target_data.shape)

print(encoder_input_data[1000])
print([tokenizer.index_word[s] for s in encoder_input_data[1000] if s != 0])

print(decoder_input_data[1000])
print([tokenizer.index_word[s] for s in decoder_input_data[1000] if s != 0])

print(decoder_target_data[1000])
print([tokenizer.index_word[s] for s in decoder_target_data[1000] if s != 0])

(24058, 1577)
(24058, 2331)
(24058, 2331)
[  92 1515 5455 ...    0    0    0]
['그럼', '애를', '낳고', '동안', '게', '아니라', '그러니까', '알바를', '하신', '거네요.']
[  3 399   0 ...   0   0   0]
['<sos>', '그렇죠']
[399   4   0 ...   0   0   0]
['그렇죠', '<eos>']


In [10]:
# 데이터 로드하는 class 정의
class QADataset(Dataset):
    def __init__(self, encoder_inputs, decoder_inputs, decoder_targets):
        super().__init__()
        self.encoder_inputs = encoder_inputs
        self.decoder_inputs = decoder_inputs
        self.decoder_targets = decoder_targets

    def __len__(self):
        return len(self.encoder_inputs)

    def __getitem__(self, index):
        return (
            torch.tensor(self.encoder_inputs[index], dtype=torch.long),
            torch.tensor(self.decoder_inputs[index], dtype=torch.long),
            torch.tensor(self.decoder_targets[index], dtype=torch.long),
        )

In [19]:
# Train-Test 데이터 분리
indices = np.arange(len(encoder_input_data))

train_index, val_index = train_test_split(
    indices,
    test_size=0.2,
    random_state=0
)

print(len(train_index), len(val_index))

train_dataset = QADataset(
    encoder_input_data[train_index],
    decoder_input_data[train_index],
    decoder_target_data[train_index]
)

val_dataset = QADataset(
    encoder_input_data[val_index],
    decoder_input_data[val_index],
    decoder_target_data[val_index]
)

19246 4812


In [20]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [21]:
# encoder class 정의
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, latent_dim, embedding_matrix=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, latent_dim, batch_first=True)

    def forward(self, X, hidden, cell):
        X = self.embedding(X)
        output, (h_s, c_s) = self.lstm(X, (hidden, cell))
        return h_s, c_s

In [ ]:
# decoder class 정의
class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, latent_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, latent_dim, batch_first=True)
        self.fc = nn.Linear(latent_dim, vocab_size)

    def forward(self, X, hidden, cell):
        X = self.embedding(X)
        output, (h_s, c_s) = self.lstm(X, (hidden, cell))
        logits = self.fc(output)
        return logits, h_s, c_s

In [ ]:
# seq2seq class 정의
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, source, target):
        h_s, c_s = self.encoder(source)
        output, h_s, c_s = self.decoder(target, h_s, c_s)
        return output

In [26]:
# encoder, decoder + seq2seq 모델 정의
encoder = Encoder(num_words, EMBEDDING_DIM, LATENT_DIM)
decoder = Decoder(num_words, EMBEDDING_DIM, LATENT_DIM)
model = Seq2Seq(encoder, decoder)

In [28]:
Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(3371, 100, padding_idx=0)
    (lstm): LSTM(100, 512, batch_first=True)
  )
  (decoder): Decoder(
    (embedding): Embedding(8874, 100, padding_idx=0)
    (lstm): LSTM(100, 512, batch_first=True)
    (fc): Linear(in_features=512, out_features=8874, bias=True)
  )
)

SyntaxError: invalid syntax (17003640.py, line 2)

In [ ]:
# 분류형이기 때문에 지표로써 CrossEntropyLoss 사용
# 최적화 함수: Adam 사용
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

encoder = Encoder(num_words, EMBEDDING_DIM, LATENT_DIM)
decoder = Decoder(num_words, EMBEDDING_DIM, LATENT_DIM)
model = Seq2Seq(encoder, decoder).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.AdamW(model.parameters(), lr=0.001)

train_losses, train_accs = [], []
val_losses, val_accs = [], []

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

encoder = Encoder(num_words, EMBEDDING_DIM, LATENT_DIM)
decoder = Decoder(num_words, EMBEDDING_DIM, LATENT_DIM)
model = Seq2Seq(encoder, decoder).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.AdamW(model.parameters(), lr=0.001)

train_losses, train_accs, val_losses, val_accs = [], [], [], []

for epoch in range(EPOCHS):
  model.train()
  train_loss, train_correct, train_tokens = 0, 0, 0

  for enc_inputs, dec_inputs, dec_targets in train_loader:
    enc_inputs = enc_inputs.to(device)
    dec_inputs = dec_inputs.to(device)
    dec_targets = dec_targets.to(device)

    optimizer.zero_grad()

    # teacher forcing
    output = model(enc_inputs, dec_inputs)
    output = output.view(-1, output.size(-1))
    dec_targets = dec_targets.view(-1)

    loss = criterion(output, dec_targets)
    loss.backward()
    optimizer.step()

    preds = output.argmax(dim=-1)
    train_loss += loss.detach().cpu().item()
    mask = dec_targets != 0
    correct = (preds == dec_targets) & mask
    train_correct += correct.sum().detach().cpu().item()
    train_tokens += mask.sum().detach().cpu().item()

  train_loss /= len(train_loader)
  train_acc = train_correct / train_tokens
  train_losses.append(train_loss)
  train_accs.append(train_acc)

  model.eval()
  with torch.no_grad():
    val_loss, val_correct, val_tokens = 0, 0, 0

    for enc_inputs, dec_inputs, dec_targets in val_loader:
      enc_inputs = enc_inputs.to(device)
      dec_inputs = dec_inputs.to(device)
      dec_targets = dec_targets.to(device)

      output = model(enc_inputs, dec_inputs)
      output = output.view(-1, output.size(-1))
      dec_targets = dec_targets.view(-1)

      loss = criterion(output, dec_targets)

      preds = output.argmax(dim=-1)
      val_loss += loss.detach().cpu().item()
      mask = dec_targets != 0
      correct = (preds == dec_targets) & mask
      val_correct += correct.sum().detach().cpu().item()
      val_tokens += mask.sum().detach().cpu().item()

    val_loss /= len(val_loader)
    val_acc = val_correct / val_tokens
    val_losses.append(val_loss)
    val_accs.append(val_acc)

  print(f'Epoch {epoch+1}/{EPOCHS} TrainLoss={train_loss:.4f} TrainAcc={train_acc:.4f} ValLoss={val_loss:.4f} ValAcc={val_acc:.4f}')

In [ ]:
model = torch.load('seq2seq_teacher_forcing.pth', weights_only=False)
model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def translate(input_seq, model, kor_tokenizer, max_len=max_encoder_len, device=device):
  model = model.to(device)
  model.eval()
  encoder = model.encoder
  decoder = model.decoder

  input_seq = torch.tensor(input_seq, dtype=torch.long).to(device)

  # Encoder 처리
  with torch.no_grad():
    hidden, cell = encoder(input_seq)

  # Decoder 출력(Auto Regressive)
  sos_index = kor_tokenizer.word_index['']
  eos_index = kor_tokenizer.word_index['']

  output_sentences = []

  target_seq = torch.tensor([[sos_index]], dtype=torch.long).to(device)

  with torch.no_grad():
    for _ in range(max_len):
      output, hidden, cell = decoder(target_seq, hidden, cell)  # (batch_size, seq_len, vocab_size)
      proba = output.squeeze(1).softmax(dim=-1)                 # (batch_size, vocab_size)
      pred = proba.argmax(dim=-1).detach().cpu().item()

      if pred == eos_index:
        break

      if pred > 0:
        word = kor_tokenizer.index_word[pred]
        output_sentences.append(word)

      target_seq = torch.tensor([[pred]], dtype=torch.long).to(device)

  return ' '.join(output_sentences)

In [ ]:
import numpy as np

for _ in range(5):
  index = np.random.choice(len())
  input_seq = encoder_input_data[index:index+1]
  output = translate(input_seq, model, tokenizer)

  print(f'질문: {encoder_input_data[index]}')
  print(f'응답: {[index]}')
  print(f'모델 추론 응답: {output}')
  print()
     

In [ ]:
import torch
from tensorflow.keras.preprocessing.sequence import pad_sequences

def generate_answer(input_text, model, tokenizer, max_encoder_len, max_decoder_len, device):
    model.eval()

    idx2word = tokenizer.index_word
    sos_idx = tokenizer.word_index.get('<sos>')
    eos_idx = tokenizer.word_index.get('<eos>')
    unk_idx = tokenizer.word_index.get('<unk>')

    if sos_idx is None or eos_idx is None:
        return "특수 토큰(<sos>, <eos>)이 정의되지 않았어요."

    input_seq = tokenizer.texts_to_sequences([input_text])
    print("입력 시퀀스:", input_seq)   # 디버깅용

    input_seq = pad_sequences(
        input_seq,
        maxlen=max_encoder_len,
        padding='post',
        truncating='post'
    )
    input_seq = torch.tensor(input_seq, dtype=torch.long).to(device)

    with torch.no_grad():
        hidden, cell = model.encoder(input_seq)

    target_seq = torch.tensor([[sos_idx]], dtype=torch.long).to(device)
    output_words = []

    with torch.no_grad():
        for _ in range(max_decoder_len):
            output, hidden, cell = model.decoder(target_seq, hidden, cell)
            pred_idx = output[:, -1, :].argmax(dim=-1).item()

            print("예측 idx:", pred_idx, "예측 단어:", idx2word.get(pred_idx, '<unk>'))  # 디버깅용

            if pred_idx == eos_idx or pred_idx == 0:
                break

            if unk_idx is not None and pred_idx == unk_idx:
                break

            pred_word = idx2word.get(pred_idx, '<unk>')

            if pred_word not in ['<sos>', '<eos>', '<unk>']:
                output_words.append(pred_word)

            target_seq = torch.tensor([[pred_idx]], dtype=torch.long).to(device)

    if len(output_words) == 0:
        return "적절한 답변을 생성하지 못했어요."

    return ' '.join(output_words)


def ask_chatbot(input_text):
    return generate_answer(
        input_text=input_text,
        model=model,
        tokenizer=tokenizer,
        max_encoder_len=max_encoder_len,
        max_decoder_len=max_decoder_len,
        device=device
    )

In [ ]:
while True:
    user_input = input("질문: ").strip()

    if user_input in ['종료']:
        print("챗봇: 대화를 종료합니다.")
        break

    if user_input == "":
        print("챗봇: 질문을 입력해 주세요.")
        continue

    answer = ask_chatbot(user_input)
    print("챗봇:", answer)